In [10]:
# Step 1: Configuration ("wrds_sql" for live WRDS connection or "local_files" for CSV exports)
import pathlib

import numpy as np
import pandas as pd
import wrds

DATA_SOURCE_MODE = "wrds_sql"
WRDS_USERNAME = "your_wrds_username"
# In this case we download the WRDS datasets via SQL queries and save them as CSV files in the "data/wrds"
# It from 2013-01-01 to 2026-12-31, which is the range of dates we are interested in for our analysis.
START_DATE = "2013-01-01"
END_DATE = "2026-12-31"

# "mid_month" (~15th settlement, known before month-end) or "month_end"
SHORT_INTEREST_SETTLEMENT = "mid_month"

WRDS_RAW_DIR = pathlib.Path("data/wrds")
PROCESSED_DIR = pathlib.Path("data/processed")
WRDS_RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_SHORT_INTEREST_FILE = WRDS_RAW_DIR / "compustat_short_interest.csv.gz"
LOCAL_CCM_LINK_FILE = WRDS_RAW_DIR / "ccm_link_table.csv.gz"
LOCAL_CRSP_MONTHLY_FILE = WRDS_RAW_DIR / "crsp_monthly.csv.gz"

OUTPUT_SHORT_INTEREST_FILE = PROCESSED_DIR / "wrds_monthly_short_interest.csv.gz"
OUTPUT_CRSP_MONTHLY_FILE = PROCESSED_DIR / "wrds_crsp_monthly.csv.gz"

In [11]:
# Step 2: Standardize Compustat short interest and link (gvkey, iid) to CRSP permno via CCM
def normalize_compustat_iid(raw_iid: pd.Series) -> pd.Series:
    cleaned = raw_iid.fillna("").astype(str).str.strip().str.upper()
    cleaned = cleaned.str.replace(r"\.0$", "", regex=True)
    cleaned = cleaned.replace({"NAN": "", "NONE": "", "<NA>": ""})
    needs_zero_pad = (cleaned.str.len() == 1) & cleaned.str.isdigit()
    return cleaned.mask(needs_zero_pad, cleaned.str.zfill(2))


def read_wrds_csv_preserving_ids(csv_path: pathlib.Path) -> pd.DataFrame:
    header_cols = pd.read_csv(csv_path, nrows=0).columns
    id_names = {"gvkey", "iid", "liid", "cusip", "ncusip", "hdrcusip", "cusip8"}
    str_dtypes = {
        str(col): str for col in header_cols if str(col).strip().lower() in id_names
    }
    return pd.read_csv(csv_path, dtype=str_dtypes, low_memory=False)


def collapse_single_short_interest_feed(
    raw_si: pd.DataFrame,
    settlement_rule: str,
) -> pd.DataFrame:
    si_df = raw_si.copy()
    si_df.columns = [str(col).strip().lower() for col in si_df.columns]

    si_df["gvkey"] = (
        si_df["gvkey"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(6)
    )
    if "iid" in si_df.columns:
        si_df["iid"] = normalize_compustat_iid(si_df["iid"])
    else:
        si_df["iid"] = ""

    si_df["datadate"] = pd.to_datetime(si_df["datadate"], errors="coerce")
    si_df["shortint"] = pd.to_numeric(si_df["shortint"], errors="coerce")
    if "shortintadj" in si_df.columns:
        si_df["shortintadj"] = pd.to_numeric(si_df["shortintadj"], errors="coerce")
    else:
        si_df["shortintadj"] = np.nan

    si_df = si_df.dropna(subset=["gvkey", "datadate", "shortint"]).copy()
    si_df = si_df.loc[si_df["shortint"] >= 0].copy()
    si_df["month_end"] = si_df["datadate"] + pd.offsets.MonthEnd(0)
    si_df = si_df.sort_values(["gvkey", "iid", "month_end", "datadate"])

    # Adjust mid-month shortint if a stock split occurs between mid-month and month-end
    has_adj = (
        si_df["shortint"].notna()
        & (si_df["shortint"] > 0)
        & si_df["shortintadj"].notna()
        & (si_df["shortintadj"] > 0)
    )
    si_df["cum_adj_factor"] = np.where(has_adj, si_df["shortintadj"] / si_df["shortint"], np.nan)

    last_in_month = si_df.drop_duplicates(subset=["gvkey", "iid", "month_end"], keep="last")
    month_end_factors = last_in_month[["gvkey", "iid", "month_end", "cum_adj_factor"]].rename(
        columns={"cum_adj_factor": "end_cum_adj_factor"}
    )

    keep_side = "first" if settlement_rule == "mid_month" else "last"
    monthly_df = si_df.drop_duplicates(
        subset=["gvkey", "iid", "month_end"], keep=keep_side
    ).copy()

    if settlement_rule == "mid_month":
        monthly_df = pd.merge(
            monthly_df, month_end_factors, on=["gvkey", "iid", "month_end"], how="left"
        )
        valid_ratio = (
            monthly_df["cum_adj_factor"].notna()
            & monthly_df["end_cum_adj_factor"].notna()
            & (monthly_df["end_cum_adj_factor"] > 0)
        )
        within_month_split = np.where(
            valid_ratio,
            monthly_df["cum_adj_factor"] / monthly_df["end_cum_adj_factor"],
            1.0,
        )
        monthly_df["shortint"] = monthly_df["shortint"] * within_month_split

    out_cols = ["gvkey", "iid", "month_end", "datadate", "shortint", "shortintadj"]
    return monthly_df[out_cols].reset_index(drop=True)


def collapse_short_interest_monthly(
    raw_si: pd.DataFrame,
    settlement_rule: str,
) -> pd.DataFrame:
    col_map = {str(col).strip().lower(): str(col) for col in raw_si.columns}
    if "from_legacy" not in col_map:
        return collapse_single_short_interest_feed(raw_si, settlement_rule)

    flag_col = col_map["from_legacy"]
    flag_num = pd.to_numeric(raw_si[flag_col], errors="coerce").fillna(0)
    feed_frames: list[pd.DataFrame] = []
    for flag_val in [1, 0]:
        sub_raw = raw_si.loc[flag_num == flag_val]
        if sub_raw.empty:
            continue
        sub_monthly = collapse_single_short_interest_feed(sub_raw, settlement_rule)
        sub_monthly["from_legacy"] = flag_val
        feed_frames.append(sub_monthly)

    combined_si = pd.concat(feed_frames, ignore_index=True)
    combined_si = combined_si.sort_values(
        ["gvkey", "iid", "month_end", "from_legacy"], ascending=[True, True, True, False]
    )
    combined_si = combined_si.drop_duplicates(
        subset=["gvkey", "iid", "month_end"], keep="first"
    )
    return combined_si.drop(columns=["from_legacy"]).reset_index(drop=True)


def link_short_interest_to_permno(
    monthly_si: pd.DataFrame,
    raw_ccm: pd.DataFrame,
) -> pd.DataFrame:
    ccm = raw_ccm.copy()
    ccm.columns = [str(col).strip().lower() for col in ccm.columns]

    if "lpermno" in ccm.columns and "permno" not in ccm.columns:
        ccm = ccm.rename(columns={"lpermno": "permno"})

    ccm["gvkey"] = (
        ccm["gvkey"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(6)
    )
    ccm["liid"] = normalize_compustat_iid(ccm["liid"]) if "liid" in ccm.columns else ""
    ccm["permno"] = pd.to_numeric(ccm["permno"], errors="coerce")
    ccm["linktype"] = ccm["linktype"].fillna("").astype(str).str.strip().str.upper()
    ccm["linkprim"] = ccm["linkprim"].fillna("").astype(str).str.strip().str.upper()
    ccm["linkdt"] = pd.to_datetime(ccm["linkdt"], errors="coerce").fillna(
        pd.Timestamp("1900-01-01")
    )
    ccm["linkenddt"] = pd.to_datetime(ccm["linkenddt"], errors="coerce").fillna(
        pd.Timestamp("2099-12-31")
    )

    valid_ccm = (
        ccm["permno"].notna()
        & ccm["linktype"].isin(["LC", "LU"])
        & ccm["linkprim"].isin(["P", "C"])
    )
    ccm = ccm.loc[valid_ccm].copy()
    ccm["permno"] = ccm["permno"].astype("int64")
    ccm["prim_rank"] = np.where(ccm["linkprim"] == "P", 0, 1)
    ccm["type_rank"] = np.where(ccm["linktype"] == "LC", 0, 1)

    si_work = monthly_si.copy()
    si_work["si_row_id"] = np.arange(len(si_work), dtype="int64")

    # Match on gvkey within active CCM link dates and require exact issue match (iid == liid)
    candidates = pd.merge(si_work, ccm, on="gvkey", how="inner")
    in_date_window = (candidates["datadate"] >= candidates["linkdt"]) & (
        candidates["datadate"] <= candidates["linkenddt"]
    )
    candidates = candidates.loc[in_date_window].copy()

    iid_matches = (candidates["iid"] == candidates["liid"]) | (candidates["iid"] == "") | (
        candidates["liid"] == ""
    )
    candidates = candidates.loc[iid_matches].copy()
    candidates["exact_iid_rank"] = np.where(
        (candidates["iid"] != "") & (candidates["iid"] == candidates["liid"]), 0, 1
    )

    candidates = candidates.sort_values(
        ["si_row_id", "exact_iid_rank", "prim_rank", "type_rank", "permno"]
    )
    best_links = candidates.drop_duplicates(subset=["si_row_id"], keep="first").copy()
    best_links = best_links.sort_values(
        ["permno", "month_end", "exact_iid_rank", "prim_rank", "type_rank"]
    )
    best_links = best_links.drop_duplicates(subset=["permno", "month_end"], keep="first")

    out_cols = ["permno", "gvkey", "iid", "month_end", "datadate", "shortint", "shortintadj"]
    return best_links[out_cols].sort_values(["permno", "month_end"]).reset_index(drop=True)

In [12]:
# Step 3: Standardize CRSP monthly stock data (supports both CIZ v2 and Legacy SIZ)
def standardize_crsp_monthly(raw_crsp: pd.DataFrame) -> pd.DataFrame:
    crsp = raw_crsp.copy()
    crsp.columns = [str(col).strip().lower() for col in crsp.columns]

    is_ciz_v2 = ("mthcaldt" in crsp.columns) or ("hdrcusip" in crsp.columns)

    if is_ciz_v2:
        crsp["date"] = pd.to_datetime(crsp["mthcaldt"], errors="coerce")
        crsp["prc_abs"] = pd.to_numeric(crsp["mthprc"], errors="coerce").abs()
        crsp["ret"] = pd.to_numeric(crsp["mthret"], errors="coerce")
        crsp["shrout_thousands"] = pd.to_numeric(crsp["shrout"], errors="coerce")

        if "mthcumfacshr" in crsp.columns:
            crsp["cfacshr"] = pd.to_numeric(crsp["mthcumfacshr"], errors="coerce")
        elif "cfacshr" in crsp.columns:
            crsp["cfacshr"] = pd.to_numeric(crsp["cfacshr"], errors="coerce")
        else:
            crsp["cfacshr"] = np.nan

        # Historical point-in-time CUSIP is CUSIP in CIZ v2 (HdrCUSIP is header-only)
        hist_cusip = crsp["cusip"].fillna("")
        if "hdrcusip" in crsp.columns:
            hist_cusip = hist_cusip.where(hist_cusip.str.strip() != "", crsp["hdrcusip"].fillna(""))
        crsp["cusip8"] = hist_cusip.astype(str).str.strip().str.upper().str[:8]

        is_common = (
            (crsp["sharetype"].fillna("").str.upper() == "NS")
            & (crsp["securitytype"].fillna("").str.upper() == "EQTY")
            & (crsp["securitysubtype"].fillna("").str.upper() == "COM")
            & (crsp["usincflg"].fillna("").str.upper() == "Y")
            & (crsp["issuertype"].fillna("").str.upper().isin(["ACOR", "CORP"]))
        )
        is_major_exch = crsp["primaryexch"].fillna("").str.upper().isin(["N", "A", "Q"])
        if "conditionaltype" in crsp.columns and "tradingstatusflg" in crsp.columns:
            is_major_exch = (
                is_major_exch
                & (crsp["conditionaltype"].fillna("").str.upper() == "RW")
                & (crsp["tradingstatusflg"].fillna("").str.upper() == "A")
            )
        crsp["common_stock"] = is_common
        crsp["major_exchange"] = is_major_exch
        crsp["is_nyse"] = crsp["primaryexch"].fillna("").str.upper() == "N"
    else:
        crsp["date"] = pd.to_datetime(crsp["date"], errors="coerce")
        crsp["prc_abs"] = pd.to_numeric(crsp["prc"], errors="coerce").abs()
        crsp["ret"] = pd.to_numeric(crsp["ret"], errors="coerce")
        crsp["shrout_thousands"] = pd.to_numeric(crsp["shrout"], errors="coerce")
        crsp["cfacshr"] = (
            pd.to_numeric(crsp["cfacshr"], errors="coerce")
            if "cfacshr" in crsp.columns
            else np.nan
        )

        # Historical point-in-time CUSIP is NCUSIP in Legacy SIZ (CUSIP is header-only)
        if "ncusip" in crsp.columns:
            hist_cusip = crsp["ncusip"].fillna("")
            if "cusip" in crsp.columns:
                hist_cusip = hist_cusip.where(
                    hist_cusip.str.strip() != "", crsp["cusip"].fillna("")
                )
        else:
            hist_cusip = crsp["cusip"].fillna("")
        crsp["cusip8"] = hist_cusip.astype(str).str.strip().str.upper().str[:8]

        shrcd_num = pd.to_numeric(crsp["shrcd"], errors="coerce")
        exchcd_num = pd.to_numeric(crsp["exchcd"], errors="coerce")
        crsp["common_stock"] = shrcd_num.isin([10, 11])
        crsp["major_exchange"] = exchcd_num.isin([1, 2, 3])
        crsp["is_nyse"] = exchcd_num == 1

    crsp["permno"] = pd.to_numeric(crsp["permno"], errors="coerce")
    crsp = crsp.dropna(subset=["permno", "date", "shrout_thousands"]).copy()
    crsp = crsp.loc[crsp["shrout_thousands"] > 0].copy()
    crsp["permno"] = crsp["permno"].astype("int64")

    # Convert CRSP shrout from thousands to exact shares
    crsp["shares_out"] = crsp["shrout_thousands"] * 1000.0
    crsp["mktcap_millions"] = (crsp["prc_abs"] * crsp["shares_out"]) / 1_000_000.0
    crsp["month_end"] = crsp["date"] + pd.offsets.MonthEnd(0)

    crsp = crsp.sort_values(["permno", "month_end", "date"])
    crsp = crsp.drop_duplicates(subset=["permno", "month_end"], keep="last")

    keep_cols = [
        "permno",
        "month_end",
        "date",
        "cusip8",
        "prc_abs",
        "ret",
        "shares_out",
        "mktcap_millions",
        "cfacshr",
        "common_stock",
        "major_exchange",
        "is_nyse",
    ]
    return crsp[keep_cols].reset_index(drop=True)

In [13]:
# Step 4: Retrieve WRDS data (splicing sec_shortint_legacy + sec_shortint) and save outputs
# Note: WRDS may call you for verification
def load_wrds_datasets_via_sql(
    username: str,
    start_date: str,
    end_date: str,
    settlement_rule: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    db = wrds.Connection(wrds_username=username)
    try:
        comp_tables = set(db.list_tables(library="comp") or [])
        crsp_tables = set(db.list_tables(library="crsp") or [])

        # Prefer sec_shortint_legacy through Dec 2024 and splice sec_shortint for 2025+
        raw_si_parts: list[pd.DataFrame] = []
        if "sec_shortint_legacy" in comp_tables:
            q_legacy = f"""
                SELECT gvkey, iid, datadate, shortint, shortintadj
                FROM comp.sec_shortint_legacy
                WHERE datadate BETWEEN '{start_date}' AND '{end_date}'
                  AND shortint IS NOT NULL
            """
            legacy_raw = db.raw_sql(q_legacy)
            legacy_raw["from_legacy"] = 1
            raw_si_parts.append(legacy_raw)

        if "sec_shortint" in comp_tables:
            q_current = f"""
                SELECT gvkey, iid, datadate, shortint, shortintadj
                FROM comp.sec_shortint
                WHERE datadate BETWEEN '{start_date}' AND '{end_date}'
                  AND shortint IS NOT NULL
            """
            current_raw = db.raw_sql(q_current)
            current_raw["from_legacy"] = 0
            raw_si_parts.append(current_raw)

        if not raw_si_parts:
            raise RuntimeError("Neither comp.sec_shortint_legacy nor comp.sec_shortint found.")

        si_raw = pd.concat(raw_si_parts, ignore_index=True)

        ccm_sql = """
            SELECT gvkey, liid, lpermno AS permno, linktype, linkprim, linkdt, linkenddt
            FROM crsp.ccmxpf_lnkhist
            WHERE linktype IN ('LC', 'LU')
              AND linkprim IN ('P', 'C')
        """
        ccm_raw = db.raw_sql(ccm_sql)

        crsp_start = (pd.Timestamp(start_date) - pd.DateOffset(months=6)).strftime("%Y-%m-%d")
        if "msf_v2" in crsp_tables:
            crsp_sql = f"""
                SELECT permno, mthcaldt, mthprc, mthret, shrout, mthcumfacshr,
                       sharetype, securitytype, securitysubtype, usincflg,
                       issuertype, primaryexch, conditionaltype, tradingstatusflg,
                       cusip, hdrcusip
                FROM crsp.msf_v2
                WHERE mthcaldt BETWEEN '{crsp_start}' AND '{end_date}'
            """
            crsp_raw = db.raw_sql(crsp_sql)
        else:
            crsp_sql = f"""
                SELECT a.permno, a.date, a.prc, a.ret, a.shrout, a.cfacshr,
                       b.ncusip, b.cusip, b.shrcd, b.exchcd, c.dlret
                FROM crsp.msf AS a
                LEFT JOIN crsp.msenames AS b
                  ON a.permno = b.permno
                 AND b.namedt <= a.date
                 AND a.date <= b.nameendt
                LEFT JOIN crsp.msedelist AS c
                  ON a.permno = c.permno
                 AND date_trunc('month', a.date) = date_trunc('month', c.dlstdt)
                WHERE a.date BETWEEN '{crsp_start}' AND '{end_date}'
            """
            crsp_raw = db.raw_sql(crsp_sql)
            ret_num = pd.to_numeric(crsp_raw["ret"], errors="coerce")
            dlret_num = pd.to_numeric(crsp_raw["dlret"], errors="coerce")
            both_present = ret_num.notna() & dlret_num.notna()
            only_dlret = ret_num.isna() & dlret_num.notna()
            compounded = ret_num.copy()
            compounded.loc[both_present] = (
                1.0 + ret_num.loc[both_present]
            ) * (1.0 + dlret_num.loc[both_present]) - 1.0
            compounded.loc[only_dlret] = dlret_num.loc[only_dlret]
            crsp_raw["ret"] = compounded
    finally:
        db.close()

    # Save raw WRDS SQL tables to data/wrds/ for offline reuse
    si_raw.to_csv(LOCAL_SHORT_INTEREST_FILE, index=False, compression="gzip")
    ccm_raw.to_csv(LOCAL_CCM_LINK_FILE, index=False, compression="gzip")
    crsp_raw.to_csv(LOCAL_CRSP_MONTHLY_FILE, index=False, compression="gzip")
    print(f"Saved raw WRDS tables -> {WRDS_RAW_DIR}")

    combined_si = collapse_short_interest_monthly(si_raw, settlement_rule)
    linked_si = link_short_interest_to_permno(combined_si, ccm_raw)
    clean_crsp = standardize_crsp_monthly(crsp_raw)
    return linked_si, clean_crsp


if DATA_SOURCE_MODE == "wrds_sql":
    monthly_si_panel, crsp_monthly_panel = load_wrds_datasets_via_sql(
        username=WRDS_USERNAME,
        start_date=START_DATE,
        end_date=END_DATE,
        settlement_rule=SHORT_INTEREST_SETTLEMENT,
    )
elif DATA_SOURCE_MODE == "local_files":
    raw_si_local = read_wrds_csv_preserving_ids(LOCAL_SHORT_INTEREST_FILE)
    raw_ccm_local = read_wrds_csv_preserving_ids(LOCAL_CCM_LINK_FILE)
    raw_crsp_local = read_wrds_csv_preserving_ids(LOCAL_CRSP_MONTHLY_FILE)

    collapsed_si_local = collapse_short_interest_monthly(raw_si_local, SHORT_INTEREST_SETTLEMENT)
    monthly_si_panel = link_short_interest_to_permno(collapsed_si_local, raw_ccm_local)
    crsp_monthly_panel = standardize_crsp_monthly(raw_crsp_local)
else:
    raise ValueError(f"Unsupported DATA_SOURCE_MODE: {DATA_SOURCE_MODE}")

monthly_si_panel.to_csv(OUTPUT_SHORT_INTEREST_FILE, index=False, compression="gzip")
crsp_monthly_panel.to_csv(OUTPUT_CRSP_MONTHLY_FILE, index=False, compression="gzip")

print(f"Saved {len(monthly_si_panel):,} linked short-interest rows -> {OUTPUT_SHORT_INTEREST_FILE}")
print(f"Saved {len(crsp_monthly_panel):,} monthly CRSP rows -> {OUTPUT_CRSP_MONTHLY_FILE}")

WRDS recommends setting up a .pgpass file.
pgpass file created at C:\Users\Shenjian\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Saved raw WRDS tables -> data\wrds
Saved 969,717 linked short-interest rows -> data\processed\wrds_monthly_short_interest.csv.gz
Saved 1,316,824 monthly CRSP rows -> data\processed\wrds_crsp_monthly.csv.gz
